# svGrowth Architecture Breakdown

This notebook provides an **introductory guide to the svGrowth framework**, covering:
- **Architecture**: How data flows through the class hierarchy
- **Parameter loading**: Reading and inspecting G&R model configurations
- **Data access**: Where parameters live and how to retrieve them
- **Computational workflow**: Step-by-step breakdown of one complete timestep

**Who is this for?**
- New users learning the svGrowth API
- Developers planning to extend svGrowth with new features

**What you'll learn:**
1. The **separation of concerns** between data ownership (Configuration → Layer → Constituent) and computational tools (Kinetics, Mechanics)
2. How to **access and inspect** all model parameters programmatically
3. The **time-stepping algorithm** that couples mass density, geometry, and stress
4. Where to find **implementation details** for each computational step

## Notebook Structure

This notebook is organized into **four main sections**:

| Section | What You'll Learn |
|---------|-------------------|
| **1. Background** | Theoretical foundation of constrained mixture G&R models |
| **2. Architecture** | How svGrowth separates data ownership from computation |
| **3. Parameter Loading** | How to load YAML configs and access all stored data |
| **4. Timestep Walkthrough** | Detailed breakdown of the coupled mass-geometry-stress algorithm |

---
## Prerequisites

```bash
# Install svGrowth dependencies
pip install -r ../requirements.txt
```

## Setup: Import Modules

In [3]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Math

# Add src directory to path
notebook_dir = Path.cwd()
src_path = notebook_dir.parent / 'src'
sys.path.insert(0, str(src_path))

# Import svGrowth modules
from io_handler import IOHandler
from configuration import Configuration
from custom_logging import init_logging

# Printing options
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.precision', 4)

---
## 1. Background

svGrowth is based on the **constrained mixture theory**, where tissues are modeled as a mixture of multiple **structurally-significant constituents** (for example elastin, collagen, smooth muscle cells). Tissues adapt to mechanical stimuli over time through continuous turnover of their constituents. Each constituent has its own mass production/degradation kinetics and mechanical properties, but all constituents are constrained to deform together (constrained mixture). More information can be found in the following foundational paper:

> **Humphrey, J. D., & Rajagopal, K. R. (2002)**  
> *A constrained mixture model for growth and remodeling of soft tissues*  
> **Mathematical Models and Methods in Applied Sciences**, 12(03), 407-430.  
> DOI: [10.1142/S0218202502001714](https://doi.org/10.1142/S0218202502001714)

This notebook is using G&R parameters for a cerebral artery (thin wall cylinder) with three constituents (elastin, collagen, smooth muscle cells), as described in the reference paper below. While this notebook is focused on a cylindrical vessel, note that svGrowth's architecture can support other types of geometry (i.e. spheres for cardiac G&R) and eventually arbitrary 3D geometries.

> **Latorre, M., & Humphrey, J. D. (2018)**  
> *A mechanobiologically equilibrated constrained mixture model for growth and remodeling of soft tissues*  
> **ZAMM-Journal of Applied Mathematics and Mechanics**, 98, 2048–2071.  
> DOI: [10.1002/zamm.201700302](https://doi.org/10.1002/zamm.201700302)

---
## 2. svGrowth architecture

Before initializing and running a G&R simulation, let's take a look at **where data lives** in svGrowth. The code architecture consists of three main blocks: (i) simulation framework (purple), (ii) physical representation (blue), (iii) computational tools (green), as detailed in the figure below.

| Block | Responsible for | 
|-------|-----------------|
| **Simulation framework** |  Time-stepping, Numerical solvers, Convergence | 
| **Physical representation** | Data ownership, History management, Orchestrating computations|
| **Computational tools** | Performing computations (stateless)

All **solver and numerics-related parameters** are stored in the **simulation framework** block. \
All **G&R parameters** are stored in the **physical representation** block, which mirrors the physical structure of the vessel. \
The **computational tools** block is stateless, meaning that its classes **do not store any data**. All data required for a given computation (e.g. heredity integral) is provided as an input from the physical representation block, and computed results are returned without retaining any internal states or variable history data.

In the next section, we will load an input parameter file and examine in detail what these parameters are, where they are stored and how to access them.

![Data Hierarchy](../docs/svGrowth_architecture.png)

---
## 3. Loading an input parameter file and accessing stored data

All required parameters to launch a G&R simulation are inputed from a .yaml parameter file. From input parameters, a Configuration object can be initialized at the homeostatic state. Upon Configuration initilization, reference homeostatic parameters are reported for all layers and constituents, as shown below. More details about each class are found in the following subsections.

In [4]:
# Load parameters
io_handler = IOHandler()
params_file = Path('../examples/latorre2018.yaml')
params = io_handler.load_parameters(params_file)

# Initialize logging
init_logging('INFO', config_file='../src/logging_config.yaml')

# Create configuration
config = Configuration.from_parameters(params)


HOMEOSTATIC CONFIGURATION
[INFO] ┌─ Layer: Cerebral artery ──────────────────────────────────
[INFO] │  Kinematics: ThinWallKinematics
[INFO] │  Constituents: 3
[INFO] │
[INFO] │  Homeostatic Mass Density
[INFO] │    Referential mass density (ρ_h): 1050.00 kg/m³
[INFO] │
[INFO] │  Homeostatic Geometry
[INFO] │    Inner radius (a_h):       1.4000 mm
[INFO] │    Thickness (h_h):          0.1200 mm
[INFO] │    Axial stretch (λ_z_h):    1.0000
[INFO] │
[INFO] │  Homeostatic Loading
[INFO] │    Pressure (P_h):           14.18 kPa
[INFO] │    Flow rate (Q_h):          1.00 mL/s
[INFO] │    WSS (τ_w_h):              1.86 Pa
[INFO] │
[INFO] │  Homeostatic Intramural Stress (diagonal)
[INFO] │    σ_rr:                     0.00 kPa
[INFO] │    σ_θθ:                     165.40 kPa
[INFO] │    σ_zz:                     2.40 kPa
[INFO] │
[INFO] │  ┌─ Constituent: elastin ──────────────────────────
[INFO] │  │    Mass fraction:       0.02 (21.00 kg/m³)
[INFO] │  │    Deposition stretch:  [0.51, 1.4

---
### 3.1 Configuration Class

The **Configuration** object represents the entire physical geometry and acts as the top-level orchestrator. Configuration is defined as a collection of Layers (or constrained mixtures) and handles the order of operation and other interaction between Layers. In our example, the Configuration object consists of a single Layer named "Cerebral artery" as shown in the code block below.

**Key Data:**
| Property | Description |
|----------|-------------|
| `layers` | Collection of `Layer` objects |

See [`configuration.py()`](../src/configuration.py#L1) for implementation details.

In [5]:
print(f"Number of layers: {len(config.layers)}")
print(f"\nLayer names:")
for layer in config.layers:
    print(f"  - {layer.name}")

Number of layers: 1

Layer names:
  - Cerebral artery


---
### 3.2 Layer Class

A **Layer** object represents a constrained mixture of multiple structurally-significant constituents. Layer is responsible for managing its own Constituents and orchestrating all computations at the constrained mixture level (e.g. Cauchy stress computation, total referential mass density computation). All mixture-related parameters and their histories live here.

**Key Data:**
| Property | Description | Examples |
|----------|-------------|----------|
| **Constituents** | Collection of constituent objects | Elastin, collagen, smooth muscle cells|
| **Kinematics** | Geometry type and assumptions | Thin wall cylinder, thick wall sphere |
| **Geometry** | Deformation gradient and related quantities | Inner radius $a$, thickness $h$, axial stretch $\lambda_z$ |
| **External loading** | Applied loads and perturbations | Intramural pressure $P$, flow rate $Q$ |
| **Mass** | Total referential mass density | $\rho_R$ (sum of constituent densities) |
| **Stress** | Cauchy stress tensor | $\boldsymbol{\sigma}$ (function of constituent stresses) |


The code block below demonstrates how to access these quantities. 

See [`layer.py()`](../src/layer.py#L1) for implementation details.

In [6]:
# Get layer
layer = config.layers[0]

print(f"Layer: {layer.name}")
print(f"Kinematics type: {type(layer.kinematics).__name__}")
print(f"Number of constituents: {len(layer.constituents)}")

# Homeostatic (reference) state at t=0
print("\n" + "="*60)
print("Homeostatic State (t=0)")
print("="*60)

timestep_0 = 0

print(f"\nGeometry:")
print(f"  Inner radius (a_h):       {layer.get_inner_radius(timestep_0)*1000:.4f} mm")
print(f"  Thickness (h_h):          {layer.get_thickness(timestep_0)*1000:.4f} mm")
print(f"  Axial stretch (λ_z_h):    {layer.get_axial_stretch(timestep_0):.4f}")

print(f"\nLoading:")
print(f"  Pressure (P_h):           {layer.get_pressure(timestep_0)/1000:.2f} kPa")
print(f"  Flow rate (Q_h):          {layer.get_flow_rate(timestep_0):.4f} m³/day")
print(f"  WSS (τ_w_h):              {layer.get_wss(timestep_0):.4f} Pa")

print(f"\nMass:")
print(f"  Total ref. density (ρ_h): {layer.get_density(timestep_0):.2f} kg/m³")

print(f"\nCauchy stress (diagonal):")
stress_h = layer.get_stress(timestep_0)
print(f"  σ_rr: {stress_h[0,0]/1000:.2f} kPa")
print(f"  σ_θθ: {stress_h[1,1]/1000:.2f} kPa")
print(f"  σ_zz: {stress_h[2,2]/1000:.2f} kPa")

Layer: Cerebral artery
Kinematics type: ThinWallKinematics
Number of constituents: 3

Homeostatic State (t=0)

Geometry:
  Inner radius (a_h):       1.4000 mm
  Thickness (h_h):          0.1200 mm
  Axial stretch (λ_z_h):    1.0000

Loading:
  Pressure (P_h):           14.18 kPa
  Flow rate (Q_h):          0.0000 m³/day
  WSS (τ_w_h):              1.8560 Pa

Mass:
  Total ref. density (ρ_h): 1050.00 kg/m³

Cauchy stress (diagonal):
  σ_rr: 0.00 kPa
  σ_θθ: 165.40 kPa
  σ_zz: 2.40 kPa


---
### 3.3 Constituent Class

**Constituents** are the building blocks for Layers. All constituent-related parameters and their histories live here. Superscript $\alpha$ denotes a constituent.

**Key Data:**
| Property | Description | Symbol/Examples |
|----------|-------------|--------|
| **Mass** | Referential mass density | $\rho_R^\alpha$ |
| **Deposition stretch** | Stress-free configuration tensor | $\mathbf{G}^\alpha$ |
| **Constitutive model** | Stress-strain relationship | Fung exponential, neo-Hookean, etc. |
| **Kinetics** *(optional)* | Mass turnover parameters | Production rate, degradation rate, survival function |
| **Active properties** *(optional)* | Contractile cell parameters | Maximum active stress, activation dynamics |


In code blocks below, we will access constituent-related parameters for smooth muscle cells to demonstrate how to retrieve all stored properties.

See [`constituent.py`](../src/constituent.py) for implementation details.

In [7]:
# Get all constituent names
constituent_names = layer.get_constituent_names()
print(f"Constituents: {constituent_names}")

# Access consittuent by name
smc = layer.get_constituent("smooth_muscle_cells")
print(f"Retrieved constituent object ({smc}) with name '{smc.name}'")

Constituents: ['elastin', 'smooth_muscle_cells', 'collagen_circumferential']
Retrieved constituent object (<constituent.SingleConstituent object at 0x12fd6cd60>) with name 'smooth_muscle_cells'


In [8]:
# Print consituent state summary for t=0
# This function automatically accesses all constituent parameters and prints them in a nice format. 
# For manually accesing each parameter, see code blocks below.
smc.print_state()

[INFO] ┌─ Constituent: smooth_muscle_cells ──────────────
[INFO] │    Mass fraction:       0.76 (798.00 kg/m³)
[INFO] │    Deposition stretch:  [1.00, 1.30, 1.00]
[INFO] │
[INFO] │  Constitutive Model
[INFO] │    Type:                FungExponentialModel
[INFO] │    Parameters:          c1 = 10.0 kPa, c2 = 3.5
[INFO] │    Fiber orientation:   90.0° (circumferential)
[INFO] │
[INFO] │  Active Stress Properties
[INFO] │    T_act_h:             170.00 kPa
[INFO] │    k_act:               0.143 1/day
[INFO] │    λ_0:                 0.400
[INFO] │    λ_m:                 1.100
[INFO] │    CB:                  0.833
[INFO] │    CS:                  0.416
[INFO] │
[INFO] │  Kinetics - Mass Production
[INFO] │    Stimulus function:             Linear
[INFO] │    Gain parameters:               K_intramural_stress = 2.0, K_wss = -1.0
[INFO] │
[INFO] │  Kinetics - Mass Degradation
[INFO] │    Survival function:             Exponential
[INFO] │    Degradation rate function:     Quadratic
[INFO] │

In [9]:
# Access referential mass density and compute mass fraction
constituent_ref_mass_density = smc.get_rhoR_alpha(timestep=0)
layer_ref_mass_density = layer.get_density(timestep=0)
mass_fraction = constituent_ref_mass_density / layer_ref_mass_density

print("Mass Density")
print("-" * 70)
print(f"  ρ_constituent(0)  = {constituent_ref_mass_density:.2f} kg/m³")
print(f"  ρ_layer(0)        = {layer_ref_mass_density:.2f} kg/m³")
print(f"  Mass fraction     = {mass_fraction}")

Mass Density
----------------------------------------------------------------------
  ρ_constituent(0)  = 798.00 kg/m³
  ρ_layer(0)        = 1050.00 kg/m³
  Mass fraction     = 0.76


In [10]:
# Access deposition stretch tensor
G_alpha = smc.deposition_stretch
lambda_r = G_alpha[0, 0]
lambda_theta = G_alpha[1, 1]
lambda_z = G_alpha[2, 2]

print("Deposition Stretch")
print("-" * 70)
print(f"  G_α (deposition stretch tensor):")
print(f"    λ_r = {lambda_r:.2f}")
print(f"    λ_θ = {lambda_theta:.2f}")
print(f"    λ_z = {lambda_z:.2f}")
print(f"\n  Full tensor:")
print(f"{G_alpha}")

Deposition Stretch
----------------------------------------------------------------------
  G_α (deposition stretch tensor):
    λ_r = 1.00
    λ_θ = 1.30
    λ_z = 1.00

  Full tensor:
[[1.  0.  0. ]
 [0.  1.3 0. ]
 [0.  0.  1. ]]


In [11]:
# Access constitutive law parameters
model = smc.constitutive_model
model_type = model.__class__.__name__

# Extract parameters (Fung exponential model)
c1_kPa = model.c1 / 1000  # Pa → kPa
c2 = model.c2
fiber_angle = model.fiber_angle_degrees

print("Constitutive Model")
print("-" * 70)
print(f"  Type: {model_type}")
print(f"\n  Parameters:")
print(f"    c1 = {c1_kPa:.1f} kPa")
print(f"    c2 = {c2:.1f}")
print(f"\n  Fiber orientation:")
print(f"    Angle     = {fiber_angle}°")

Constitutive Model
----------------------------------------------------------------------
  Type: FungExponentialModel

  Parameters:
    c1 = 10.0 kPa
    c2 = 3.5

  Fiber orientation:
    Angle     = 90.0°


In [12]:
# Access active stress properties
active_props = smc.active_properties

# Extract and convert units
T_act_kPa = active_props['T_act'] / 1000  # Pa → kPa
k_act = active_props['k_act']             # 1/day
lambda_0 = active_props['lambda_0']       # dimensionless
lambda_m = active_props['lambda_m']       # dimensionless
CB = active_props['CB']                   # dimensionless
CS = active_props['CS']                   # dimensionless

# Access active radius history
a_act_h = smc.active_radius_history[0]

print("Active Properties")
print("-" * 70)
print(f"  Maximum active stress:")
print(f"    T_act = {T_act_kPa:.2f} kPa")
print(f"\n  Activation rate:")
print(f"    k_act = {k_act:.4f} 1/day")
print(f"\n  Activation stretch range:")
print(f"    λ_0 (minimum) = {lambda_0:.3f}")
print(f"    λ_m (optimal) = {lambda_m:.3f}")
print(f"\n  Vasomotor control:")
print(f"    CB (basal-related magnitude) = {CB:.3f}")
print(f"    CS (shear-related magnitude) = {CS:.3f}")
print(f"\n  Homeostatic active radius:")
print(f"    a_act(0) = {a_act_h*1000:.4f} mm")

Active Properties
----------------------------------------------------------------------
  Maximum active stress:
    T_act = 170.00 kPa

  Activation rate:
    k_act = 0.1429 1/day

  Activation stretch range:
    λ_0 (minimum) = 0.400
    λ_m (optimal) = 1.100

  Vasomotor control:
    CB (basal-related magnitude) = 0.833
    CS (shear-related magnitude) = 0.416

  Homeostatic active radius:
    a_act(0) = 1.4000 mm


In [13]:
# Access mass production function
prod_func = smc.kinetics.production_function
prod_type = prod_func.__class__.__name__

# Access gain parameters
gain_params = prod_func.gain_params

# Access homeostatic production rate
mR_h = smc.mR_alpha_history[0]

print("Mass Production")
print("-" * 70)
print(f"  Production function type: {prod_type}")
print(f"\n  Gain parameters:")
for param_name, value in gain_params.items():
    print(f"    K_{param_name} = {value}")
print(f"\n  Homeostatic production rate:")
print(f"    mR_α(0) = {mR_h:.4f} kg/(m³·day)")

Mass Production
----------------------------------------------------------------------
  Production function type: LinearProductionRate

  Gain parameters:
    K_intramural_stress = 2.0
    K_wss = -1.0

  Homeostatic production rate:
    mR_α(0) = 57.0000 kg/(m³·day)


In [14]:
# Access mass degradation function
deg_func = smc.kinetics.degradation_function
deg_type = deg_func.__class__.__name__

# Access homeostatic degradation rate
k_alpha_h = deg_func.k_alpha_h

# Access gain parameters
gain_params = deg_func.gain_params

print("Mass Degradation")
print("-" * 70)
print(f"  Degradation function type: {deg_type}")
print(f"\n  Homeostatic degradation rate:")
print(f"    k_α(0)    = {k_alpha_h:.4f} 1/day")
print(f"\n  Gain parameters:")
for param_name, value in gain_params.items():
    print(f"    K_{param_name} = {value}")

Mass Degradation
----------------------------------------------------------------------
  Degradation function type: QuadraticDegradationRate

  Homeostatic degradation rate:
    k_α(0)    = 0.0714 1/day

  Gain parameters:
    K_intramural_stress = 1.0


In [15]:
# Access survival function
surv_func = smc.kinetics.survival_function
surv_type = surv_func.__class__.__name__

print("Survival Function")
print("-" * 70)
print(f"  Survival function type: {surv_type}")

Survival Function
----------------------------------------------------------------------
  Survival function type: ExponentialSurvival


---
### 3.4 Simulation class

The **Simulation** class manages time-stepping and solver configurations. It orchestrates the coupling between mass computation and geometric equilibrium through fixed-point iteration. More info about the time-stepping algorithm in the next section.

**Key Data:**
| Property | Description | Examples |
|----------|-------------|----------|
| **Time discretization** | Timestep size and duration | `dt` (days), `n_days` (total simulation time) |
| **Solver settings** | Convergence criteria and algorithms | Fixed-point tolerance, equilibrium solver method |

See [`simulation.py`](../src/simulation.py) for implementation details.

In [16]:
# Functions to access simulation parameters are currently under development. 
# For now, we can directly access the parameters from the input file dictionary.
sim_params = params['simulation']

print("Simulation Parameters:")
print(f"  Name: {sim_params['simulation_name']}")
print(f"  Time step (dt): {sim_params['dt']} days")
print(f"  Total days: {sim_params['n_days']} days")
print(f"  Integration method: {sim_params['integration_method']}")
print(f"\nFixed-point solver:")
for key, val in sim_params.get('fixed_point_solver', {}).items():
    print(f"  {key}: {val}")
print(f"\nEquilibrium solver:")
for key, val in sim_params.get('equilibrium_solver', {}).items():
    print(f"  {key}: {val}")

Simulation Parameters:
  Name: latorre2018
  Time step (dt): 0.5 days
  Total days: 560.0 days
  Integration method: trapezoidal

Fixed-point solver:
  tolerance: 1e-12
  max_iterations: 50

Equilibrium solver:
  method: brentq
  tolerance: 1e-5


---
# 4. Walkthrough for a single timestep

In this section, we manually step through what when advancing **from timestep 0 to timestep 1** to understand the core G&R time-stepping algorithm.

## The Coupled System

Computing G&R solutions for a given timestep involves three tightly coupled variables:

1. **Mass density**
2. **Geometry**
3. **Stress**

These variables depend on each other through:
- **Incompressibility**: Mass change drives geometry change via the volume ratio `J = ρ_h/ρ`
- **Mechanical equilibrium**: Geometry determines stress via Laplace law
- **Mechanobiological feedback**: Stress regulates mass production/degradation

**Solution strategy:**
- **Fixed-point iteration**: Iteratively update mass density and geometry-stress until convergence.
- **Root-finding**: Nested in fixed-point iteration. For a given mass, iteratively update geometry and stress until convergence.

**Notation:**
All mathematical notation follows **Latorre & Humphrey (2018)**. See the [reference paper](https://doi.org/10.1002/zamm.201700302) for complete derivations.

**Scope:**
We focus on the **main computational workflow** and key function calls. Implementation details are referenced in the corresponding classes:

| Class | Responsibility |
|-------|----------------|
| [`Simulation`](../src/simulation.py) | Time-stepping orchestration |
| [`Configuration`](../src/configuration.py) | Top-level orchestration |
| [`Layer`](../src/layer.py) | Mixture-level orchestration |
| [`Constituent`](../src/constituent.py) | Constituent-level orchestration |
| [`Kinetics`](../src/kinetics.py) | Mass production/degradation computations |
| [`Mechanics`](../src/mechanics.py) | Stress computations |
| [`FixedPointSolver`](../src/fixed_point_solver.py) | Fixed-point iteration and convergence |
| [`Solver`](../src/solver.py) | Root-finding algorithms and convergence |


![Timestep schematic](../docs/anatomy_of_timestep.png)

---
### 4.1. Make Initial Guesses

Before solving the coupled system, we initialize all variables from the previous timestep. This provides a starting point for the fixed-point iteration.

In [17]:
# Target timestep
timestep = 1
dt = sim_params['dt']
time = timestep * dt

print(f"Timestep: {timestep}")
print(f"Physical time: {time} days")
print("\n" + "="*60)
print("Making Initial Guesses")
print("="*60)

# Guess all variables
config.guess_all_rhoR_alpha(timestep, guess_method="from_previous_timestep")
config.guess_geometry(timestep, guess_method="from_previous_timestep")
config.guess_loading_variables(timestep)
config.guess_stress_and_wss(timestep)

print("\nGuessed values:")
print(f"  ρ(1) = {layer.get_density(timestep):.2f} kg/m³ (copied from t=0)")
print(f"  a(1) = {layer.get_inner_radius(timestep)*1000:.4f} mm (copied from t=0)")
print(f"  h(1) = {layer.get_thickness(timestep)*1000:.4f} mm (copied from t=0)")
print(f"  P(1) = {layer.get_pressure(timestep)/1000:.2f} kPa (copied from t=0)")

Timestep: 1
Physical time: 0.5 days

Making Initial Guesses

Guessed values:
  ρ(1) = 1050.00 kg/m³ (copied from t=0)
  a(1) = 1.4000 mm (copied from t=0)
  h(1) = 0.1200 mm (copied from t=0)
  P(1) = 14.18 kPa (copied from t=0)


---
### 4.2. Apply Perturbations

External loading changes (here pressure, flow, axial stretch) are applied via the `PerturbationManager` class. After updating loading variables, we recompute **derived quantities** that depend on them:
- **WSS** (depends on flow rate `Q` and inner radius `a`)
- **Deformation gradient F** (depends on axial stretch `λ_z`)

**Key functions:**
```python
config.apply_perturbations(timestep, time)  # Update P, Q, λ_z
config.update_wss_and_F(timestep)           # Recompute WSS, F
```
See [`perturbations.py`](../src/perturbations.py) for implementation details.

In [18]:
print("="*60)
print("Applying Perturbations")
print("="*60)

# Store pre-perturbation values
P_before = layer.get_pressure(timestep)
Q_before = layer.get_flow_rate(timestep)

# Apply perturbations
config.apply_perturbations(timestep, time)
config.update_wss_and_F(timestep)

# Store post-perturbation values
P_after = layer.get_pressure(timestep)
Q_after = layer.get_flow_rate(timestep)

print(f"\nPressure:")
print(f"  Before: {P_before/1000:.2f} kPa")
print(f"  After:  {P_after/1000:.2f} kPa")
print(f"  Change: {(P_after - P_before)/1000:.2f} kPa")

print(f"\nFlow rate:")
print(f"  Before: {Q_before:.4f} m³/day")
print(f"  After:  {Q_after:.4f} m³/day")
print(f"  Change: {Q_after - Q_before:.4f} m³/day")

# Note: At timestep=1, perturbation might not be active yet
if abs(P_after - P_before) < 1e-6:
    print("\n⚠️ No perturbation active at this timestep")

Applying Perturbations

Pressure:
  Before: 14.18 kPa
  After:  14.18 kPa
  Change: 0.00 kPa

Flow rate:
  Before: 0.0000 m³/day
  After:  0.0000 m³/day
  Change: 0.0000 m³/day

⚠️ No perturbation active at this timestep


---
### 4.3 Compute Referential Mass Density

Referential mass density evolution is computed via heredity integrals that account for continuous production and degradation of material. Mass density at the layer level is computed by summing up the mass densities of individual constituents.

$$
\begin{equation}
{\rho_R^\alpha
}(s) = 
\int_{-\infty}^{s}
\dot{m}_R^{\alpha}(\tau)\, q^{\alpha}(s,\tau)\, d\tau
\end{equation}
$$

where:
- $\dot{m}_R^{\alpha}(\tau)$ : mass production rate at time $\tau$
- $q(s,\tau)$ : survival function

Orchestration is performed by `Configuration`, `Layer` and `Constituent` classes. Computations are delegated to `Kinetics` and `Integrators` classes. 

**Hierarchical computation:**

The computation propagates through the class hierarchy:

```
Configuration.compute_all_rhoR()
  ├─> Layer.compute_rhoR()                    # Sum over constituents
  │     ├─> Constituent.compute_rhoR_alpha()  # Heredity integral
  │     │     ├─> Kinetics.compute_degradation_rate()
  │     │     ├─> Kinetics.compute_production_rate()
  │     │     ├─> Kinetics.compute_survival_function()
  │     │     └─> Kinetics.compute_heredity_integral()
  │     └─> rho_layer = Σ rhoR_alpha          
  └─> Returns list of layer densities
```

**Workflow (per constituent):**

The `Constituent.compute_rhoR_alpha()` method delegates the following computations to the `Kinetics` class:

1. Compute degradation rate $k_α(s)$ 
2. Compute Production rate $\dot{m}_R^{\alpha}(\tau)$ 
3. Compute survival function $q(s,\tau)$ 
4. Compute mass density heredity integral ${\rho_R^\alpha}(s)$

**Note:** Only constituents with `Kinetics` undergo turnover. Elastin (no kinetics) maintains constant mass.

**Implementation references:**

| Class | Function | Responsibility |
|-------|--------|----------------|
| [`Configuration`](../src/configuration.py) | `compute_all_rhoR()` | Top-level orchestrator |
| [`Layer`](../src/layer.py) | `compute_rhoR()` | Sums constituent contributions |
| [`Constituent`](../src/constituent.py) | `compute_rhoR_alpha()` | Heredity integral orchestration |
| [`Kinetics`](../src/kinetics.py) | `compute_k_alpha()`<br>`compute_production_rate()`<br>`compute_survival_function()`<br>`compute_heredity_integral()` | Mass production/degradation computations |

![Mass density schematic](../docs/ref_mass_density.png)

In [19]:
print("="*60)
print("Computing Mass Density (Iteration 1)")
print("="*60)

# Hierarchical computation:
# 1. Configuration.compute_all_rhoR() calls Layer.compute_rhoR() for each layer
# 2. Layer.compute_rhoR() calls Constituent.compute_rhoR_alpha() for each constituent
# 3. Layer sums constituent masses: ρ_layer = Σ ρ_α
# 4. Results propagate back up: Constituent → Layer → Configuration
rhoR_list = config.compute_all_rhoR(
    timestep,
    dt,
    integration_method=sim_params['integration_method'],
    survival_function_computation=sim_params['survival_function_computation']
)

print(f"\nTotal layer density: {layer.get_density(timestep):.2f} kg/m³")
print(f"\nConstituent breakdown:")

for const in layer.constituents:
    rho_alpha = const.get_rhoR_alpha(timestep)
    print(f"  {const.name}: {rho_alpha:.2f} kg/m³")

Computing Mass Density (Iteration 1)

Total layer density: 1050.01 kg/m³

Constituent breakdown:
  elastin: 21.00 kg/m³
  smooth_muscle_cells: 798.00 kg/m³
  collagen_circumferential: 231.00 kg/m³


---
### 4.4. Compute Cauchy stress

Constituent stress is computed via heredity integrals that sum contributions from all material cohorts deposited over time.

\begin{equation}
{\boldsymbol{\sigma^\alpha}}(s) = \frac{1}{\rho_R^o}
\int_{-\infty}^{s}
\dot{m}_R^{\alpha}(\tau)\, q^{\alpha}(s,\tau)\,\boldsymbol{\hat\sigma^\alpha}(s,\tau) \,d\tau
\end{equation}

where:
- $\boldsymbol{\sigma}^\alpha(s)$ : constituent Cauchy stress at current time $s$
- $\hat{\boldsymbol{\sigma}}^\alpha(s,\tau)$ : partial constituent stress for cohort deposited at time $\tau$
- $\rho_R^o$ : homeostatic referential mass density of mixture

**Hierarchical computation:**

The computation propagates through the class hierarchy:

```
Configuration.compute_all_stress()
  ├─> Layer.compute_stress()                   # Sum over constituents
  │     ├─> Constituent.compute_sigma_alpha()               # Heredity integral
  │     │     ├─> Mechanics.compute_F_alpha()               # F_α(s,τ)
  │     │     ├─> Mechanics.compute_S_hat_alpha()           # Ŝ_α
  │     │     ├─> Mechanics.compute_sigma_hat_alpha()       # σ̂_α(s,τ)
  │     │     ├─> Mechanics.integrate_constituent_stress()  # σ_α(s)
  │     └─> sigma_layer = Σ sigma_alpha           
  └─> Returns list of layer stresses
```
**Workflow (per constituent):**

The [`Constituent.compute_sigma_alpha()`](../src/constituent.py) function delegates the following computations to the `Mechanics` class:

1. Compute constituent deformation gradient $\mathbf{F}_\alpha(s,\tau)$
2. Compute 2nd Piola-Kirchhoff stress $\hat{\mathbf{S}}_\alpha$ 
3. Compute partial constituent stress $\hat{\boldsymbol{\sigma}}_\alpha(s,\tau)$ 
4. Compute constituent Cauchy stress heredity integral $\boldsymbol{\sigma}^\alpha(s)$

**Implementation references:**

| Class | Function | Responsibility |
|-------|--------|----------------|
| [`Configuration`](../src/configuration.py) | `compute_all_stress()` | Top-level orchestrator |
| [`Layer`](../src/layer.py) | `compute_stress()` | Sums constituent contributions |
| [`Constituent`](../src/constituent.py) | `compute_sigma_alpha()` | Heredity integral orchestration |
| [`Mechanics`](../src/mechanics.py) | `compute_F_alpha_for_all_cohorts()`<br>`compute_S_hat_alpha_for_all_cohorts()`<br>`compute_sigma_hat_alpha_for_all_cohorts()`<br>`integrate_constituent_stress()` | Stress computations |

![Stress schematic](../docs/cauchy_stress.png)

In [20]:
print("="*60)
print("Computing Cauchy Stress (Iteration 1)")
print("="*60)

# Hierarchical computation:
# 1. Configuration.compute_all_stress() calls Layer.compute_stress() for each layer
# 2. Layer.compute_stress() calls Constituent.compute_sigma_alpha() for each constituent
# 3. Layer sums constituent stresses: σ_layer = Σ σ_α
# 4. Results propagate back up: Constituent → Layer → Configuration

config.compute_all_stress(
    timestep,
    dt,
    integration_method=sim_params['integration_method'],
    survival_function_computation=sim_params['survival_function_computation']
)

# Display total mixture stress
sigma_total = layer.get_stress(timestep)
print(f"\nTotal mixture stress (diagonal):")
print(f"  σ_rr: {sigma_total[0,0]/1000:.4f} kPa (radial)")
print(f"  σ_θθ: {sigma_total[1,1]/1000:.4f} kPa  (circumferential)")
print(f"  σ_zz: {sigma_total[2,2]/1000:.4f} kPa  (axial)")

# Breakdown by constituent
print(f"\nConstituent contributions (note lagrange multiplier has not been applied to these values):")
print(f"\n{'Constituent':<25} {'σ_rr (kPa)':<12} {'σ_θθ (kPa)':<12} {'σ_zz (kPa)':<12}")
print("-" * 65)

for const in layer.constituents:
    sigma_alpha = const.stress_history[timestep]
    print(f"{const.name:<25} {sigma_alpha[0,0]/1000:>11.4f} {sigma_alpha[1,1]/1000:>11.4f} {sigma_alpha[2,2]/1000:>11.4f}")

Computing Cauchy Stress (Iteration 1)

Total mixture stress (diagonal):
  σ_rr: 0.0000 kPa (radial)
  σ_θθ: 165.3958 kPa  (circumferential)
  σ_zz: 2.4000 kPa  (axial)

Constituent contributions (note lagrange multiplier has not been applied to these values):

Constituent               σ_rr (kPa)   σ_θθ (kPa)   σ_zz (kPa)  
-----------------------------------------------------------------
elastin                        0.3676      2.7675      2.7675
smooth_muscle_cells            0.0000    110.1910      0.0000
collagen_circumferential       0.0000     52.8048      0.0000


---
### 4.5 Solve Geometric Equilibrium

For a given mass density, we solve for the geometry that satisfies **mechanical equilibrium** (unknown is mid-point radius):

$$
\sigma_{\theta\theta}^{\text{mixture}} = \sigma_{\theta\theta}^{\text{theoretical}}
$$

where:
- **σ_θθ (mixture)**: Computed from constituent stresses (step 4.4)
- **σ_θθ (theoretical)**: From Laplace law: `P·a/h` (thin-wall assumption)

**Solution method:**
The solver uses **Brent's method** (default root-finding) to find the mid-point radius `r_mid` where the residual is zero:

```
residual = σ_θθ(mixture) - σ_θθ(theoretical)
```

**Workflow (per solver iteration):**
For each trial `r_mid`:
1. Update geometry from incompressibility: `J = ρ_h/ρ` → compute thickenss `h`, inner radus `a`
2. Update WSS (depends on new inner radius `a`)
3. Compute mixture stress (via constituent heredity integrals)
4. Compute theoretical stress (Laplace law)
5. Check residual → adjust `r_mid` and repeat

**Implementation references:**

| Class | Function | Responsibility |
|-------|--------|----------------|
| [`Layer`](../src/layer.py) | `compute_equilibrium_residual()` | Residual function for root-finding |
| [`Solver`](../src/solver.py) | `solve()` | Root-finding algorithms |

In [21]:
result = config.solve_equilibrium_geometry(
    timestep=timestep,
    dt=dt,
    integration_method='trapezoidal',
    survival_function_computation='backward',
    solver_method='brentq',      
    tolerance=1e-5,             
    verbose=False
)

print(f"Converged: {result['all_converged']}")
print(f"Iterations: {result['layer_results'][0]['iterations']}")

print(f"\nSolution:")
print(f"  Mid-radius (r_mid): {result['layer_results'][0]['solution']*1000:.4f} mm")
print(f"  Inner radius (a):   {layer.get_inner_radius(timestep)*1000:.4f} mm")
print(f"  Thickness (h):      {layer.get_thickness(timestep)*1000:.4f} mm")

Converged: True
Iterations: 6

Solution:
  Mid-radius (r_mid): 1.4600 mm
  Inner radius (a):   1.4000 mm
  Thickness (h):      0.1200 mm
